In [2]:
pip install requests beautifulsoup4

In [17]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
headers = {"User-Agent": "Mozilla/5.0"}
def scrape_article(url, category):
    resp = requests.get(url, headers = headers)
    soup = BeautifulSoup(resp.text, 'html.parser')

    heading_tag = soup.find("h1")
    heading = heading_tag.get_text(strip=True) if heading_tag else None

    paragraphs = soup.find_all("p")


    paragraphs = soup.find_all("p")

    subheading = None

    if len(paragraphs) > 0:
        subheading = paragraphs[0].get_text(strip=True)


    
    subheading = None
    
    if len(paragraphs) > 1:
        subheading = paragraphs[1].get_text(strip=True)

    return {'heading':heading, 'subheading':subheading,'category':category}

data = scrape_article("https://www.bbc.com/business","business")
print(data)

{'heading': 'Business', 'subheading': "The UK's AI Safety Institute said recent behaviour from Anthropic and OpenAI models was malicious and unprecedented.", 'category': 'business'}


In [18]:
def get_article_links(category_url):
    resp = requests.get(category_url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a['href']
        if '/news/' in href and href.startswith("/"):
            links.add("https://www.bbc.com" + href)
    return list(links)

In [ ]:
categories = {
    "business": "https://www.bbc.com/news/business",
    "politics": "https://www.bbc.com/news/politics",
    "technology": "https://www.bbc.com/news/technology",
}

rows = []
for category, url in categories.items():
    links = get_article_links(url)
    for link in links[:80]:  # limit per category
        try:
            rows.append(scrape_article(link, category))
            time.sleep(1)  # be polite, avoid hammering the server
        except Exception as e:
            print(f"Failed on {link}: {e}")

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("bbc_news_dataset.csv", index=False)

In [ ]:
df.head()
df.tail()

In [ ]:
print(df["category"].unique())

In [ ]:
print(df["category"].value_counts())

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df["text"] = df["heading"] + " " + df["subheading"]

In [ ]:
X = df["text"]
y = df["category"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)


In [ ]:
model = MultinomialNB()

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
new_article = ["Google launches a new AI-powered search engine"]

new_article = vectorizer.transform(new_article)

prediction = model.predict(new_article)

print("Predicted Category:", prediction[0])